# 📅 Interactive Date Range Calendar
Hover over any event pill to see full details.

In [6]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from datetime import date
import calendar

## ⚙️ Configuration — edit only this cell

In [7]:
# ── DATE RANGE ──────────────────────────────────────────────
START_DATE = '2026-07-09'
END_DATE   = '2026-08-02'

# ── EVENTS  {date: [(label, time, category, description)]} ──
# time: 'HH:MM' or None for all-day

EVENTS = {
    '2026-07-09': [('Llegada a GT',     '21:00',    'personal',  'Llegada a aeropuerto de Guatemala')],
    '2026-07-21': [('Dentista',     None,    'doctors',  'Cita con dentista para limpieza, revision y blanqueamiento')],
    '2026-07-22': [('Graduacion',     None,    'events',  'Graduacion de Michelle en URL')],
    '2026-07-25': [('Fiesta de graduacion',     None,    'events',  'Fiesta de graduacion de Michelle')],
    '2026-08-02': [('Salida de GT',     '22:50',    'personal',  'Salida de aeropuerto de Guatemala')],
}

# ── COLORS ──────────────────────────────────────────────────
CATEGORY_COLORS = {
    'work':     '#378ADD',
    'personal': '#1D9E75',
    'doctors': '#D85A30',
    'events':  '#7F77DD',
}

MONTHS_PER_ROW = 2
CELL_HEIGHT    = 0.9

## 🔧 Builder

In [8]:
def get_months_in_range(start, end):
    months, y, m = [], start.year, start.month
    while (y, m) <= (end.year, end.month):
        months.append((y, m))
        m += 1
        if m > 12: m, y = 1, y + 1
    return months

def axis_ref(idx):
    """Return plotly axis suffix: '' for index 1, '2' for index 2, etc."""
    return '' if idx == 1 else str(idx)

def build_month(year, month, start, end, events, today, axis_idx):
    """
    Returns (shapes, annotations, traces_with_axes, total_h).
    traces_with_axes is a list of (trace, row, col) tuples.
    """
    cal_weeks  = calendar.monthcalendar(year, month)
    n_weeks    = len(cal_weeks)
    month_name = calendar.month_name[month]
    HEADER_H   = 0.70
    DAYLAB_H   = 0.45
    TOTAL_H    = HEADER_H + DAYLAB_H + n_weeks * CELL_HEIGHT
    suf        = axis_ref(axis_idx)
    xr, yr     = f'x{suf}', f'y{suf}'

    shapes, annotations, traces_with_axes = [], [], []

    # ── month header bar ──────────────────────────────────────
    shapes.append(dict(
        type='rect', xref=xr, yref=yr,
        x0=0, x1=7, y0=TOTAL_H - HEADER_H, y1=TOTAL_H,
        fillcolor='#2C2C2A', line_width=0, layer='below'
    ))
    annotations.append(dict(
        xref=xr, yref=yr, x=3.5, y=TOTAL_H - HEADER_H / 2,
        text=f'<b>{month_name} {year}</b>',
        font=dict(color='white', size=14), showarrow=False
    ))

    # ── weekday labels ────────────────────────────────────────
    for i, d in enumerate(['Mon','Tue','Wed','Thu','Fri','Sat','Sun']):
        color = '#A32D2D' if i >= 5 else '#555553'
        annotations.append(dict(
            xref=xr, yref=yr,
            x=i + 0.5, y=TOTAL_H - HEADER_H - DAYLAB_H / 2,
            text=f'<b>{d}</b>', font=dict(color=color, size=10), showarrow=False
        ))

    # ── day cells ─────────────────────────────────────────────
    for week_idx, week in enumerate(cal_weeks):
        yb = TOTAL_H - HEADER_H - DAYLAB_H - (week_idx + 1) * CELL_HEIGHT
        yt = yb + CELL_HEIGHT

        for dow, day in enumerate(week):
            x0, x1 = dow, dow + 1

            if day == 0:
                shapes.append(dict(
                    type='rect', xref=xr, yref=yr,
                    x0=x0+0.03, x1=x1-0.03, y0=yb+0.03, y1=yt-0.03,
                    fillcolor='#EFEFEF', line=dict(color='#D3D1C7', width=0.5), layer='below'
                ))
                continue

            cell_date  = date(year, month, day)
            in_range   = start <= cell_date <= end
            is_today   = cell_date == today
            is_weekend = dow >= 5
            ds         = cell_date.strftime('%Y-%m-%d')
            day_evts   = events.get(ds, [])

            bg  = '#F8F8F6' if in_range else '#EFEFEF'
            ec  = '#378ADD' if is_today else '#D3D1C7'
            elw = 1.5 if is_today else 0.5
            if is_today: bg = '#E6F1FB'

            shapes.append(dict(
                type='rect', xref=xr, yref=yr,
                x0=x0+0.03, x1=x1-0.03, y0=yb+0.03, y1=yt-0.03,
                fillcolor=bg, line=dict(color=ec, width=elw), layer='below'
            ))

            num_color = '#A32D2D' if is_weekend else '#444441'
            if not in_range: num_color = '#B4B2A9'
            if is_today:     num_color = '#185FA5'
            annotations.append(dict(
                xref=xr, yref=yr, x=x0+0.5, y=yt - 0.17,
                text=f'<b>{day}</b>' if is_today else str(day),
                font=dict(color=num_color, size=10), showarrow=False
            ))

            # ── event pills ───────────────────────────────────
            for e_idx, evt in enumerate(day_evts[:2]):
                label = evt[0]; time = evt[1]; cat = evt[2]
                desc  = evt[3] if len(evt) == 4 else None
                color = CATEGORY_COLORS.get(cat, '#888')
                py    = yt - 0.42 - e_idx * 0.32

                shapes.append(dict(
                    type='rect', xref=xr, yref=yr,
                    x0=x0+0.08, x1=x1-0.08, y0=py-0.13, y1=py+0.13,
                    fillcolor=color, line_width=0, layer='above'
                ))

                pill_text = f'{time} {label}' if time else label
                hover_lines = [
                    f'<b>{label}</b>',
                    f'📅 {ds}',
                    f'🕐 {time}' if time else '📌 All day',
                    f'🏷 {cat.capitalize()}',
                ]
                if desc: hover_lines.append(f'📝 {desc}')

                # Store row/col alongside the trace — NOT inside go.Scatter
                row_num = (axis_idx - 1) // MONTHS_PER_ROW + 1
                col_num = (axis_idx - 1) %  MONTHS_PER_ROW + 1

                trace = go.Scatter(
                    x=[x0 + 0.5], y=[py],
                    mode='text',
                    text=[f'<span style="color:white;font-size:7px">{pill_text}</span>'],
                    textposition='middle center',
                    hovertemplate='<br>'.join(hover_lines) + '<extra></extra>',
                    hoverlabel=dict(
                        bgcolor='white', bordercolor=color,
                        font=dict(size=12, color='#2C2C2A')
                    ),
                    showlegend=False
                )
                traces_with_axes.append((trace, row_num, col_num))

            if len(day_evts) > 2:
                annotations.append(dict(
                    xref=xr, yref=yr, x=x0+0.5, y=yb+0.12,
                    text=f'+{len(day_evts)-2}',
                    font=dict(color='#888', size=8), showarrow=False
                ))

    return shapes, annotations, traces_with_axes, TOTAL_H

print('✅ Builder ready.')

✅ Builder ready.


## 📅 Render

In [9]:
start  = pd.to_datetime(START_DATE).date()
end    = pd.to_datetime(END_DATE).date()
today  = date.today()
months = get_months_in_range(start, end)
n      = len(months)
cols   = min(MONTHS_PER_ROW, n)
rows   = (n + cols - 1) // cols

fig = make_subplots(rows=rows, cols=cols,
                    horizontal_spacing=0.04,
                    vertical_spacing=0.06)

all_shapes, all_annotations = [], []

for i, (year, month) in enumerate(months):
    axis_idx = i + 1          # 1-based subplot index
    r = (i // cols) + 1
    c = (i %  cols) + 1

    shapes, annotations, traces_with_axes, total_h = build_month(
        year, month, start, end, EVENTS, today, axis_idx
    )
    all_shapes      += shapes
    all_annotations += annotations

    # ✅ row and col go into fig.add_trace(), NOT into go.Scatter()
    for trace, tr, tc in traces_with_axes:
        fig.add_trace(trace, row=tr, col=tc)

    suf = axis_ref(axis_idx)
    fig.update_layout(**{
        f'xaxis{suf}': dict(range=[0,7], showgrid=False, zeroline=False,
                            showticklabels=False, fixedrange=True),
        f'yaxis{suf}': dict(range=[0,total_h], showgrid=False, zeroline=False,
                            showticklabels=False, fixedrange=True),
    })

# hide unused subplot slots
for j in range(n, rows * cols):
    suf = axis_ref(j + 1)
    fig.update_layout(**{
        f'xaxis{suf}': dict(visible=False),
        f'yaxis{suf}': dict(visible=False),
    })

# legend swatches
for cat, color in CATEGORY_COLORS.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=12, color=color, symbol='square'),
        name=cat.capitalize(), showlegend=True
    ))

fig.update_layout(
    title=dict(
        text=f'<b>{start.strftime("%B %d, %Y")}  →  {end.strftime("%B %d, %Y")}</b>',
        x=0.5, font=dict(size=16, color='#2C2C2A')
    ),
    shapes=all_shapes,
    annotations=all_annotations,
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=rows * 430 + 80,
    margin=dict(l=20, r=20, t=60, b=60),
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.04,
                font=dict(size=12)),
    hovermode='closest',
)

fig.show()
fig.write_html('calendar_output.html')
print(f'✅ Done  |  {n} month(s)  |  {sum(len(v) for v in EVENTS.values())} events')

✅ Done  |  2 month(s)  |  5 events


## 📋 Event summary table

In [10]:
rows_data = []
for ds, evts in sorted(EVENTS.items()):
    d = pd.to_datetime(ds).date()
    if start <= d <= end:
        for evt in evts:
            label, time, cat = evt[0], evt[1], evt[2]
            desc = evt[3] if len(evt) == 4 else ''
            rows_data.append({
                'Date':        ds,
                'Day':         d.strftime('%A'),
                'Time':        time if time else 'All day',
                'Event':       label,
                'Category':    cat.capitalize(),
                'Description': desc or '—'
            })

df_events = pd.DataFrame(rows_data)
print(f'Events within range ({START_DATE} → {END_DATE}):')
df_events

Events within range (2026-07-09 → 2026-08-02):


,Date,Day,Time,Event,Category,Description
0,2026-07-09,Thursday,21:00,Llegada a GT,Personal,Llegada a aeropuerto de Guatemala
1,2026-07-21,Tuesday,All day,Dentista,Doctors,"Cita con dentista para limpieza, revision y bl..."
2,2026-07-22,Wednesday,All day,Graduacion,Events,Graduacion de Michelle en URL
3,2026-07-25,Saturday,All day,Fiesta de graduacion,Events,Fiesta de graduacion de Michelle
4,2026-08-02,Sunday,22:50,Salida de GT,Personal,Salida de aeropuerto de Guatemala
